# 23 — NLP Evaluation & Error Analysis

**Learning objective.** Match metrics to task structure and turn mistakes into actionable error categories.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.metrics import precision_recall_fscore_support, classification_report
truth=['pos','pos','neg','neg','neutral','neutral']
pred =['pos','neg','neg','neg','neutral','pos']
print(classification_report(truth,pred,zero_division=0))

              precision    recall  f1-score   support

         neg       0.67      1.00      0.80         2
     neutral       1.00      0.50      0.67         2
         pos       0.50      0.50      0.50         2

    accuracy                           0.67         6
   macro avg       0.72      0.67      0.66         6
weighted avg       0.72      0.67      0.66         6



In [3]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
ref='the model generated a concise summary'.split()
hyp='the model produced a concise summary'.split()
bleu=sentence_bleu([ref],hyp,smoothing_function=SmoothingFunction().method1)
def rouge1_f1(ref,hyp):
    from collections import Counter
    a,b=Counter(ref),Counter(hyp); overlap=sum((a&b).values())
    p=overlap/max(1,len(hyp)); r=overlap/max(1,len(ref));
    return 0 if p+r==0 else 2*p*r/(p+r)
print('BLEU:',round(bleu,3),'ROUGE-1 F1:',round(rouge1_f1(ref,hyp),3))

BLEU: 0.254 ROUGE-1 F1: 0.833


In [4]:
errors=pd.DataFrame({
 'text':['not good','Apple support is slow','refund not received'],
 'gold':['neg','neg','billing'],
 'pred':['pos','neutral','technical'],
 'error_type':['negation','entity/domain ambiguity','label boundary']})
errors

                    text     gold       pred               error_type
0               not good      neg        pos                 negation
1  Apple support is slow      neg    neutral  entity/domain ambiguity
2    refund not received  billing  technical           label boundary

A metric is not the product objective. Classification, generation, retrieval and extraction need different metrics; slice analysis (language, length, channel, geography, product, time) is often more useful than one aggregate score.

---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Choose task-appropriate metrics
- Turn errors into categories that suggest data/model/product actions